In [0]:
from pyspark.sql.functions import *
from datetime import datetime
from zoneinfo import ZoneInfo

In [0]:
EVENT_NAME = "onlinesales"
EVENT_HUB_HOST = "onlineeventspace.servicebus.windows.net"
CONNECTION_STRING = dbutils.secrets.get(scope="onlineretail",key="onlinespace")

procuder = {
    "kafka.bootstrap.servers":f"{EVENT_HUB_HOST}:9093",
    "subscribe": EVENT_NAME,
    "kafka.security.protocol":"SASL_SSL",
    "kafka.sasl.mechanism":"PLAIN",
    'kafka.sasl.jaas.config':f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{CONNECTION_STRING}";',
    "startingOffsets": "latest",
    "failOnDataLoss": "false",
}

spark.conf.set(
    "fs.azure.account.key.onlinestorage2026.dfs.core.windows.net",
    dbutils.secrets.get(scope="onlineretail", key="storagekey")
)

In [0]:
PIPELINE_VERSION = "V-"+datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d-%H-%M-%S")
print(PIPELINE_VERSION)

In [0]:
storage = "abfss://onlinestore@onlinestorage2026.dfs.core.windows.net/"
bronze_path = storage + f"bronze/{PIPELINE_VERSION}"
bronze_cpkt = storage + f"bronze/{PIPELINE_VERSION}/checkpoint"

In [0]:
raw_df = (
    spark.readStream
    .format("kafka")
    .options(**procuder)
    .load()
)

In [0]:
json_df = (
    raw_df.selectExpr("CAST(value as string) as raw_json")

)

In [0]:
bronze_storage = (
    json_df.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_cpkt)
    .outputMode("append")
    .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .start(bronze_path)
)

In [0]:
bronze = spark.read.format("delta").load(bronze_path)
bronze.display()